# SignLingo AI Mini — Training Notebook

**Objective:** Build a small context-aware ASL pose retrieval system.

> ⚠️ **Research prototype.** Retrieved poses are from the How2Sign dataset.
> This system does retrieval (matching), not novel ASL generation.
> Poses shown are the **closest dataset match** to your sentence, not certified ASL translation.

---
### What this notebook does:
1. Installs libraries
2. Downloads a small How2Sign sample (or uses your data)
3. Loads and validates pose files
4. Builds sentence embeddings with `sentence-transformers`
5. Creates context-aware retrieval index
6. Evaluates text-only vs context-aware retrieval
7. Saves the index for local inference

In [ ]:
# STEP 1: Install libraries
!pip install -q sentence-transformers numpy pandas scikit-learn matplotlib

In [ ]:
# STEP 2: Mount Google Drive (if your data is there)
# Skip this if you upload data directly
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except ImportError:
    print('Not in Colab — skipping drive mount.')

## Dataset Setup

**How to get How2Sign data:**
1. Register at https://how2sign.github.io/
2. Download the **2D Body+Hand Pose (Mediapipe)** files
3. Each sentence clip has a `.npy` file with shape `[T, 274]` (T frames × 274 coords)
4. Download `how2sign_realigned_train.csv` for the sentence list

**For this notebook, we demonstrate with synthetic data.**
Replace the synthetic data section with your real data paths.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List

print('Imports OK')

In [ ]:
# STEP 3: Configure paths
# ========================================================
# REAL DATA: Update these paths to point to your files
METADATA_CSV = 'data/metadata.csv'          # your metadata CSV
POSE_DIR     = 'data/poses/'               # directory of .npy pose files
USE_SYNTHETIC = True   # Set False when you have real How2Sign data
NUM_SYNTHETIC = 30     # Number of synthetic samples for testing only
# ========================================================

Path('data/poses').mkdir(parents=True, exist_ok=True)
Path('models').mkdir(exist_ok=True)

In [ ]:
# STEP 4: Load or generate dataset
@dataclass
class PoseSample:
    sample_id: str
    sentence: str
    video_id: str
    narrative_id: str
    start_time: float
    end_time: float
    pose: Optional[np.ndarray]  # [T, 274]
    prev_sample_id: Optional[str] = None
    prev_sentence: Optional[str] = None
    is_synthetic: bool = False

if USE_SYNTHETIC:
    print('=' * 60)
    print('WARNING: Using SYNTHETIC random data for demonstration.')
    print('Synthetic poses are random numbers, NOT real ASL.')
    print('Replace USE_SYNTHETIC=False with real How2Sign data.')
    print('=' * 60)
    
    sentences = [
        'Hello, how are you today?',
        'My name is Sam.',
        'I am learning American Sign Language.',
        'Thank you for your patience.',
        'Please show me the sign for water.',
        'I do not understand.',
        'Can you sign that again?',
        'What is your name?',
        'Nice to meet you.',
        'See you tomorrow.',
        'The weather is nice today.',
        'I need help with this.',
        'Where is the library?',
        'How do you sign this word?',
        'I am a student.',
    ] * 2  # duplicate to reach 30 samples
    
    rng = np.random.default_rng(42)
    samples = []
    for i in range(min(NUM_SYNTHETIC, len(sentences))):
        T = rng.integers(20, 60)
        pose = rng.uniform(-0.5, 0.5, (T, 274)).astype(np.float32)
        # Center on 0.5 (simulate normalised coords)
        pose = pose + 0.5
        pose = np.clip(pose, 0, 1)
        samples.append(PoseSample(
            sample_id=f'syn_{i:03d}',
            sentence=sentences[i],
            video_id=f'video_{i//5}',
            narrative_id=f'narrative_{i//10}',
            start_time=float(i * 5),
            end_time=float(i * 5 + 4),
            pose=pose,
            prev_sample_id=f'syn_{i-1:03d}' if i > 0 else None,
            prev_sentence=sentences[i-1] if i > 0 else None,
            is_synthetic=True,
        ))
    print(f'Generated {len(samples)} synthetic samples.')
else:
    # Load real data
    df = pd.read_csv(METADATA_CSV)
    id_to_row = {r['sample_id']: r for _, r in df.iterrows()}
    samples = []
    missing = 0
    for _, row in df.iterrows():
        pose_path = Path(POSE_DIR) / f"{row['sample_id']}.npy"
        pose = None
        if pose_path.exists():
            arr = np.load(str(pose_path))
            if arr.ndim == 2 and arr.shape[1] == 274:
                pose = arr.astype(np.float32)
            else:
                print(f'  Bad shape {arr.shape} for {row["sample_id"]}')
        else:
            missing += 1
        prev_id = row.get('prev_sample_id', '')
        prev_sent = id_to_row.get(prev_id, {}).get('sentence', '') if prev_id else None
        samples.append(PoseSample(
            sample_id=row['sample_id'],
            sentence=row['sentence'],
            video_id=row.get('video_id', ''),
            narrative_id=row.get('narrative_id', ''),
            start_time=float(row.get('start_time', 0)),
            end_time=float(row.get('end_time', 0)),
            pose=pose,
            prev_sample_id=prev_id if prev_id else None,
            prev_sentence=prev_sent,
        ))
    print(f'Loaded {len(samples)} samples. Missing poses: {missing}.')

print('Sample check:')
for s in samples[:3]:
    print(f'  {s.sample_id}: "{s.sentence[:40]}" | pose={s.pose.shape if s.pose is not None else None}')

In [ ]:
# STEP 5: Build sentence embeddings
from sentence_transformers import SentenceTransformer

print('Loading sentence-transformer (all-MiniLM-L6-v2)…')
model = SentenceTransformer('all-MiniLM-L6-v2')

# Only index samples that have poses
pose_samples = [s for s in samples if s.pose is not None]
print(f'Samples with pose: {len(pose_samples)}')

if not pose_samples:
    raise ValueError('No samples with pose data. Check POSE_DIR or set USE_SYNTHETIC=True.')

sentences_list = [s.sentence for s in pose_samples]
embeddings = model.encode(sentences_list, convert_to_numpy=True, show_progress_bar=True)

# Normalize
norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9
embeddings = embeddings / norms

print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# STEP 6: Build pose feature summaries
def summarize_pose(pose: np.ndarray, dim: int = 32) -> np.ndarray:
    """Compress pose [T,274] to fixed vector via temporal mean+std."""
    subset = pose[:, :dim]
    return np.concatenate([subset.mean(axis=0), subset.std(axis=0)]).astype(np.float32)

# Build index
index = []
for s, emb in zip(pose_samples, embeddings):
    index.append({
        'sample_id': s.sample_id,
        'sentence': s.sentence,
        'narrative_id': s.narrative_id,
        'text_emb': emb,
        'pose_feat': summarize_pose(s.pose),
        'pose': s.pose,
    })

print(f'Index built with {len(index)} entries.')

In [ ]:
# STEP 7: Retrieval functions
MATCH_THRESHOLD = 0.30
POSE_WEIGHT = 0.25
TEXT_WEIGHT = 0.75

def retrieve_text_only(query: str) -> dict:
    q_emb = model.encode([query], convert_to_numpy=True)[0]
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-9)
    scores = [float(np.dot(q_emb, e['text_emb'])) for e in index]
    best_i = int(np.argmax(scores))
    best_score = scores[best_i]
    if best_score < MATCH_THRESHOLD:
        return {'match': None, 'score': best_score, 'mode': 'text_only',
                'reason': f'Score {best_score:.2f} below threshold.'}
    return {'match': index[best_i]['sample_id'], 'sentence': index[best_i]['sentence'],
            'score': best_score, 'mode': 'text_only'}

def retrieve_context_aware(query: str, prev_sentence: str = None, prev_pose: np.ndarray = None) -> dict:
    q_emb = model.encode([query], convert_to_numpy=True)[0]
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-9)
    
    if prev_sentence:
        ctx_query = f"{prev_sentence} [SEP] {query}"
        ctx_emb = model.encode([ctx_query], convert_to_numpy=True)[0]
        ctx_emb = ctx_emb / (np.linalg.norm(ctx_emb) + 1e-9)
    else:
        ctx_emb = q_emb
    
    if prev_pose is not None:
        prev_feat = summarize_pose(prev_pose)
        prev_feat = prev_feat / (np.linalg.norm(prev_feat) + 1e-9)
    else:
        prev_feat = None
    
    scores = []
    for e in index:
        txt_score = TEXT_WEIGHT * float(np.dot(q_emb, e['text_emb'])) + \
                    (1 - TEXT_WEIGHT) * float(np.dot(ctx_emb, e['text_emb']))
        if prev_feat is not None:
            pf = e['pose_feat'] / (np.linalg.norm(e['pose_feat']) + 1e-9)
            pose_score = float(np.dot(prev_feat, pf))
            score = (1 - POSE_WEIGHT) * txt_score + POSE_WEIGHT * pose_score
        else:
            score = txt_score
        scores.append(score)
    
    best_i = int(np.argmax(scores))
    best_score = scores[best_i]
    if best_score < MATCH_THRESHOLD:
        return {'match': None, 'score': best_score, 'mode': 'context_aware',
                'reason': f'Score {best_score:.2f} below threshold.'}
    return {'match': index[best_i]['sample_id'], 'sentence': index[best_i]['sentence'],
            'score': best_score, 'mode': 'context_aware'}

print('Retrieval functions defined.')

In [ ]:
# STEP 8: Leave-one-out evaluation
print('Running leave-one-out evaluation on training set...')
print('NOTE: This is in-sample evaluation only. Independent test evaluation pending real data.')

results = {'text_only': {'hits': 0, 'scores': []}, 'context_aware': {'hits': 0, 'scores': []}}

for i, s in enumerate(pose_samples):
    # Text-only
    r = retrieve_text_only(s.sentence)
    results['text_only']['scores'].append(r['score'])
    if r.get('sentence') == s.sentence:
        results['text_only']['hits'] += 1
    
    # Context-aware
    prev_pose_arr = pose_samples[i-1].pose if i > 0 else None
    r2 = retrieve_context_aware(s.sentence, s.prev_sentence, prev_pose_arr)
    results['context_aware']['scores'].append(r2['score'])
    if r2.get('sentence') == s.sentence:
        results['context_aware']['hits'] += 1

n = len(pose_samples)
print('\n--- Evaluation Results ---')
for mode, res in results.items():
    hit1 = res['hits'] / n if n else 0
    mean_score = np.mean(res['scores'])
    print(f'{mode:20s}: hit@1={hit1:.3f}, mean_score={mean_score:.3f}')

if USE_SYNTHETIC:
    print('\n⚠ Results above are on SYNTHETIC data — not meaningful for real ASL evaluation.')

In [ ]:
# STEP 9: Visualise a sample retrieval
import matplotlib.pyplot as plt

test_queries = [
    'Hello, how are you?',
    'What is your name?',
    'Thank you very much.',
]

print('\nSample retrievals:')
for q in test_queries:
    r1 = retrieve_text_only(q)
    r2 = retrieve_context_aware(q)
    print(f"  Query: '{q}'")
    print(f"    text_only:     {r1.get('sentence', 'NO MATCH')[:50]} (score={r1['score']:.3f})")
    print(f"    context_aware: {r2.get('sentence', 'NO MATCH')[:50]} (score={r2['score']:.3f})")
    print()

In [ ]:
# STEP 10: Save the retrieval index
index_data = {
    'model_name': 'all-MiniLM-L6-v2',
    'index_with_pose': index,
    'eval_results': {
        mode: {
            'hit_at_1': res['hits'] / n if n else 0,
            'mean_score': float(np.mean(res['scores'])),
            'n': n,
            'note': 'In-sample leave-one-out. Independent evaluation pending.',
        }
        for mode, res in results.items()
    },
    'is_synthetic': USE_SYNTHETIC,
}

with open('models/retrieval_index.pkl', 'wb') as f:
    pickle.dump(index_data, f)
print('Index saved to models/retrieval_index.pkl')

# Also save eval as JSON
with open('models/eval_results.json', 'w') as f:
    json.dump(index_data['eval_results'], f, indent=2)
print('Eval results saved to models/eval_results.json')

In [ ]:
# STEP 11: Download for local inference
try:
    from google.colab import files
    print('Downloading retrieval_index.pkl...')
    files.download('models/retrieval_index.pkl')
    files.download('models/eval_results.json')
    print('Place both files in SignLingo-AI-Mini/models/')
except ImportError:
    print('Not in Colab. Files saved to models/ locally.')
    print('Copy models/retrieval_index.pkl to your SignLingo-AI-Mini/models/ directory.')

## Next Steps

1. Download `models/retrieval_index.pkl` and place it in `SignLingo-AI-Mini/models/`
2. Start the backend: `uvicorn backend.main:app --reload --port 8000`
3. Start the frontend: `npm run dev` (from `frontend/`)
4. Open http://localhost:5173
5. Upload a video + transcript and click **Generate ASL Poses**

**For real data:** Set `USE_SYNTHETIC = False`, provide `METADATA_CSV` and `POSE_DIR`, then rerun.